In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from datasets.drive_dataset import DriveDataset
from models.mini_unet import MiniUNet


def train():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device)

    train_dataset = DriveDataset(
        root_dir=r"E:\CS\PostG\个人\unet_learn\DRIVE\training",
        image_size=256
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=2,
        shuffle=True
    )

    model = MiniUNet(in_channels=3, num_classes=1).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    epochs = 10

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for images, masks in train_loader:
            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            loss = criterion(logits, masks)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        print(f"Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.4f}")

    torch.save(model.state_dict(), "mini_unet_drive.pth")
    print("model saved to mini_unet_drive.pth")


if __name__ == "__main__":
    train()

device: cuda
input: torch.Size([2, 3, 256, 256])
x1 / inc: torch.Size([2, 64, 256, 256])
x2 / down1: torch.Size([2, 128, 128, 128])
x3 / down2: torch.Size([2, 256, 64, 64])
x4 / bottleneck: torch.Size([2, 512, 32, 32])
up1: torch.Size([2, 256, 64, 64])
up2: torch.Size([2, 128, 128, 128])
up3: torch.Size([2, 64, 256, 256])
out: torch.Size([2, 1, 256, 256])
input: torch.Size([2, 3, 256, 256])
x1 / inc: torch.Size([2, 64, 256, 256])
x2 / down1: torch.Size([2, 128, 128, 128])
x3 / down2: torch.Size([2, 256, 64, 64])
x4 / bottleneck: torch.Size([2, 512, 32, 32])
up1: torch.Size([2, 256, 64, 64])
up2: torch.Size([2, 128, 128, 128])
up3: torch.Size([2, 64, 256, 256])
out: torch.Size([2, 1, 256, 256])
input: torch.Size([2, 3, 256, 256])
x1 / inc: torch.Size([2, 64, 256, 256])
x2 / down1: torch.Size([2, 128, 128, 128])
x3 / down2: torch.Size([2, 256, 64, 64])
x4 / bottleneck: torch.Size([2, 512, 32, 32])
up1: torch.Size([2, 256, 64, 64])
up2: torch.Size([2, 128, 128, 128])
up3: torch.Size([2, 6

In [1]:
import torch
from torch.utils.data import DataLoader, random_split

from datasets.drive_dataset import DriveDataset
from models.mini_unet import MiniUNet
from utils.losses import BCEDiceLoss
from utils.metrics import binary_iou

from pathlib import Path
import csv


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_iou = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)
        loss = criterion(logits, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_iou += binary_iou(logits, masks)

    return total_loss / len(loader), total_iou / len(loader)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_iou = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)
        loss = criterion(logits, masks)

        total_loss += loss.item()
        total_iou += binary_iou(logits, masks)

    return total_loss / len(loader), total_iou / len(loader)


def main():

    # ====== log / checkpoint ======
    Path("logs").mkdir(exist_ok=True)
    Path("checkpoints").mkdir(exist_ok=True)

    log_path = Path("logs/train_log.csv")
    best_model_path = Path("checkpoints/best_mini_unet_drive.pth")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device)

    # ====== dataset ======
    dataset = DriveDataset(
        root_dir=r"E:\CS\PostG\个人\unet_learn\DRIVE\training",
        image_size=256
    )

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size

    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

    train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False)

    # ====== model ======
    model = MiniUNet(in_channels=3, num_classes=1).to(device)

    criterion = BCEDiceLoss(bce_weight=0.5, dice_weight=0.5)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    # ====== CSV ======
    with open(log_path, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch",
            "train_loss",
            "train_iou",
            "val_loss",
            "val_iou",
            "best_val_iou"
        ])

        epochs = 50
        best_val_iou = 0.0

        for epoch in range(epochs):

            train_loss, train_iou = train_one_epoch(
                model, train_loader, criterion, optimizer, device
            )

            val_loss, val_iou = validate(
                model, val_loader, criterion, device
            )

            if val_iou > best_val_iou:
                best_val_iou = val_iou
                torch.save(model.state_dict(), best_model_path)
                save_msg = " | saved best"
            else:
                save_msg = ""

            writer.writerow([
                epoch + 1,
                train_loss,
                train_iou,
                val_loss,
                val_iou,
                best_val_iou
            ])

            print(
                f"Epoch [{epoch+1}/{epochs}] "
                f"Train Loss: {train_loss:.4f} "
                f"Train IoU: {train_iou:.4f} "
                f"Val Loss: {val_loss:.4f} "
                f"Val IoU: {val_iou:.4f}"
                f"{save_msg}"
            )

    print("training finished")
    print("best val iou:", best_val_iou)
    print("log saved to:", log_path)
    print("best model saved to:", best_model_path)


if __name__ == "__main__":
    main()

device: cuda
Epoch [1/50] Train Loss: 0.8249 Train IoU: 0.1159 Val Loss: 0.7902 Val IoU: 0.0812 | saved best
Epoch [2/50] Train Loss: 0.7483 Train IoU: 0.1459 Val Loss: 0.7954 Val IoU: 0.0812
Epoch [3/50] Train Loss: 0.6934 Train IoU: 0.2195 Val Loss: 0.7808 Val IoU: 0.0052
Epoch [4/50] Train Loss: 0.6531 Train IoU: 0.2976 Val Loss: 0.7510 Val IoU: 0.0039
Epoch [5/50] Train Loss: 0.6217 Train IoU: 0.3659 Val Loss: 0.7066 Val IoU: 0.0705
Epoch [6/50] Train Loss: 0.6016 Train IoU: 0.3977 Val Loss: 0.6693 Val IoU: 0.1204 | saved best
Epoch [7/50] Train Loss: 0.5875 Train IoU: 0.4187 Val Loss: 0.6371 Val IoU: 0.3548 | saved best
Epoch [8/50] Train Loss: 0.5754 Train IoU: 0.4314 Val Loss: 0.6221 Val IoU: 0.4256 | saved best
Epoch [9/50] Train Loss: 0.5668 Train IoU: 0.4459 Val Loss: 0.6081 Val IoU: 0.3802
Epoch [10/50] Train Loss: 0.5602 Train IoU: 0.4655 Val Loss: 0.5950 Val IoU: 0.4200
Epoch [11/50] Train Loss: 0.5525 Train IoU: 0.4701 Val Loss: 0.5925 Val IoU: 0.4111
Epoch [12/50] Train 